# Chẩn đoán — chạy liền, KHÔNG cần GPU

Bốn mục, tất cả chỉ đọc file. Chạy hết mất dưới 2 phút.

| Mục | Trả lời câu hỏi |
|---|---|
| **A** | Số hạng loss nào đang phân kỳ? |
| **B** | Augmentation và LR schedule thực sự đang dùng là gì? |
| **C** | 9 file `val_predictions.npz` có hợp lệ không? |
| **D** | `help`/`hurt` bằng bao nhiêu → GO/NO-GO của Tầng 2 |

Chạy từ trên xuống, rồi gửi lại **toàn bộ** output.

In [ ]:
import os, sys, json, re, itertools
from pathlib import Path
from collections import defaultdict
import numpy as np

try:
    from google.colab import drive
    if not Path('/content/drive').exists(): drive.mount('/content/drive')
except ImportError: pass

SYM    = Path('/content/drive/MyDrive/Sym')
RUNS   = SYM/'runs_retry1'
SPLITS = SYM/'splits'
VARIANTS = ['baseline','current','full']
FOLDS, SEEDS = [1,2,3], [0,1,2]
AU_NAMES = ['AU1','AU2','AU4','AU6','AU9','AU12','AU25','AU26']

def find_npz(d, kind):
    if not d.exists(): return None
    for pat in (f'{kind}_predictions.npz', f'*{kind}*.npz'):
        h = sorted(d.glob(pat))
        if h: return h[0]
    return None

def run_state(v, f, s):
    out = RUNS/v/f'fold{f}'/f'seed{s}'
    return dict(out=out, ckpt=out/'best.pt',
                test=find_npz(out/'evaluation','test'),
                val =find_npz(out/'evaluation_val','val') or find_npz(out/'evaluation','val'))

def load_npz(p):
    d = np.load(p, allow_pickle=True); return {k: d[k] for k in d.files}

def subjects_of(d):
    if 'subjects' in d: return np.asarray(d['subjects']).astype(str)
    if 'paths' in d:    return np.array([str(p).split('/')[0] for p in d['paths']])
    return None

def hist(v, f, s=0):
    return json.loads((RUNS/v/f'fold{f}'/f'seed{s}'/'history.json').read_text(encoding='utf-8'))

print('RUNS   :', RUNS, '->', 'OK' if RUNS.is_dir() else 'THIẾU')
print('SPLITS :', SPLITS, '->', 'OK' if SPLITS.is_dir() else 'THIẾU')
n_val = sum(1 for v in VARIANTS for f in FOLDS for s in SEEDS if run_state(v,f,s)['val'] is not None)
n_ck  = sum(1 for v in VARIANTS for f in FOLDS for s in SEEDS if run_state(v,f,s)['ckpt'].exists())
print(f'\nrun có best.pt: {n_ck}   run có val npz: {n_val}')

## A. Số hạng nào phân kỳ?

`total` của `current` tăng 1.8×, của `full` tăng 2.6× qua 20 epoch. Ô này tách ra xem thủ phạm là `global_con`, `node_con` hay `cross_con`.

In [ ]:
for f in FOLDS:
    print('\n' + '='*78)
    print(f'FOLD {f}')
    print('='*78)
    for v in VARIANTS:
        try: h = hist(v, f)
        except Exception as e: print(f'  {v}: {e}'); continue
        keys = list(h[0]['train'].keys())
        print(f'\n--- {v} ---')
        print('  ep  ' + '  '.join(f'{k:>11}' for k in keys))
        idx = sorted({0, 1, 4, 9, 14, len(h)-1})
        for i in idx:
            e = h[i]
            print(f"  {e['epoch']:>2}  " + '  '.join(f"{e['train'][k]:>11.5f}" for k in keys))
        # xu hướng từng số hạng
        print('  xu hướng ep1 -> cuối:')
        for k in keys:
            a, b = h[0]['train'][k], h[-1]['train'][k]
            if abs(a) < 1e-12 and abs(b) < 1e-12:
                tag = 'tắt (=0)'
            elif b > a:
                tag = f'TĂNG {b/max(a,1e-12):.2f}x   <-- phân kỳ'
            else:
                tag = f'giảm {a/max(b,1e-12):.0f}x'
            print(f'      {k:<12} {a:>10.5f} -> {b:>10.5f}   {tag}')

## B. Augmentation và LR schedule đang dùng

Đọc từ `source/` được lưu kèm mỗi run — tức đúng phiên bản code đã chạy, không phải bản hiện tại trong repo.

In [ ]:
SRC = RUNS/'baseline'/'fold2'/'seed0'/'source'
print('source/ :', 'OK' if SRC.is_dir() else 'THIẾU', SRC)

print('\n' + '='*78); print('B1. dataset.py — các lớp transform'); print('='*78)
ds = (SRC/'dataset.py').read_text(encoding='utf-8')
blocks = re.findall(r'(class\s+image_\w+.*?)(?=\nclass |\Z)', ds, re.S)
if blocks:
    for b in blocks: print(b[:1200]); print('-'*60)
else:
    print('không thấy class image_* — in toàn bộ dòng có transform:')
    for l in ds.splitlines():
        if any(k in l for k in ('transform','Compose','Crop','Flip','Jitter','Normalize','Resize')):
            print('   ', l.strip())

print('\n' + '='*78); print('B2. train_contrastive.py — scheduler, augmentation, regularization'); print('='*78)
tc = (SRC/'train_contrastive.py').read_text(encoding='utf-8')
for kw in ['scheduler','adjust_learning_rate','CosineAnnealing','OneCycle','LambdaLR','StepLR',
           'param_group', 'RandomResizedCrop','RandomHorizontalFlip','RandomErasing','ColorJitter',
           'RandomCrop','image_train','image_test','transform','weight_decay','Dropout','dropout',
           'label_smoothing','mixup','AdamW','SGD']:
    hits = [l.strip() for l in tc.splitlines() if kw in l]
    mark = '' if hits else '   <-- KHÔNG CÓ'
    print(f'  {kw:22s} {len(hits):>2} dòng{mark}')
    for l in hits[:3]: print(f'        {l[:110]}')

print('\n' + '='*78); print('B3. Hàm xây transform (nếu nằm trong train_contrastive.py)'); print('='*78)
for m in re.finditer(r'(def\s+\w*(?:transform|augment|loader|dataset)\w*\(.*?)(?=\ndef |\nclass |\Z)', tc, re.S):
    print(m.group(1)[:900]); print('-'*60)

## C. Kiểm toàn vẹn 9 file val

In [ ]:
problems = []
for v in VARIANTS:
    for f in FOLDS:
        for s in SEEDS:
            st = run_state(v,f,s)
            if st['val'] is None or st['test'] is None: continue
            tag = f'{v}/fold{f}/seed{s}'
            dv, dt = load_npz(st['val']), load_npz(st['test'])
            for need in ('labels','probabilities'):
                for nm,d in (('val',dv),('test',dt)):
                    if need not in d: problems.append(f'{tag}: {nm} thiếu "{need}" (có {list(d)})')
            if 'labels' not in dv or 'probabilities' not in dv: continue
            if dv['labels'].shape != dv['probabilities'].shape:
                problems.append(f'{tag}: val shape lệch {dv["labels"].shape} vs {dv["probabilities"].shape}')
            pr = dv['probabilities']
            if pr.min() < -1e-6 or pr.max() > 1+1e-6:
                problems.append(f'{tag}: val prob ngoài [0,1]: [{pr.min():.3f},{pr.max():.3f}]')
            if pr.min() >= 0.49:
                problems.append(f'{tag}: val prob >= 0.49 — DẤU HIỆU SIGMOID HAI LẦN')
            mf = json.loads((SPLITS/f'fold{f}.json').read_text(encoding='utf-8'))
            sv, stt = subjects_of(dv), subjects_of(dt)
            if sv is not None and set(map(str,mf['val_subjects'])) != set(sv):
                problems.append(f'{tag}: subject val {sorted(set(sv))} != manifest {sorted(mf["val_subjects"])}')
            if stt is not None and set(map(str,mf['test_subjects'])) != set(stt):
                problems.append(f'{tag}: subject test lệch manifest')
            if len(dv['labels']) != len(mf['val_indices']):
                problems.append(f'{tag}: val {len(dv["labels"])} frame, manifest {len(mf["val_indices"])}')
            if sv is not None and stt is not None and (set(sv) & set(stt)):
                problems.append(f'{tag}: val/test CHUNG subject — rò rỉ!')
            ck = st['ckpt'].stat().st_mtime
            for nm,p in (('val',st['val']),('test',st['test'])):
                if p.stat().st_mtime < ck:
                    problems.append(f'{tag}: {nm}.npz cũ hơn best.pt')

ex = next((run_state(v,f,s)['val'] for v in VARIANTS for f in FOLDS for s in SEEDS
           if run_state(v,f,s)['val'] is not None), None)
if ex is not None:
    d = load_npz(ex); print(f'--- {ex.relative_to(RUNS)} ---')
    for k,a in d.items(): print(f'  {k:16s} {getattr(a,"shape","?")}  {getattr(a,"dtype","?")}')
    sv = subjects_of(d)
    if sv is not None: print(f'  subject: {sorted(set(map(str,sv)))}')
    print(f'  prob min/max: {d["probabilities"].min():.4f} / {d["probabilities"].max():.4f}')

print('\n' + ('>>> KHÔNG CÓ VẤN ĐỀ' if not problems else f'>>> {len(problems)} VẤN ĐỀ:'))
for p in problems: print('   -', p)

## D. `help` / `hurt` — GO/NO-GO của Tầng 2

Con số quyết định là **số frame phân biệt** mà ít nhất một luật sửa được, không phải tổng qua các cặp (một frame bật nhầm ba AU dính tới ba cặp — cộng lại là đếm trùng).

In [ ]:
TH = 0.5
agg_e = defaultdict(lambda: dict(help=0,hurt=0,viol=0))
agg_i = defaultdict(lambda: dict(help=0,hurt=0,viol=0))
per_sub = defaultdict(int)
n_frames = n_runs = fr_help = fr_hurt = 0

for v in VARIANTS:
    for f in FOLDS:
        for s in SEEDS:
            st = run_state(v,f,s)
            if st['val'] is None: continue
            d = load_npz(st['val'])
            y, yh = (d['labels'] > 0.5), (d['probabilities'] > TH)
            sub = subjects_of(d)
            if sub is None: sub = np.array(['?']*len(y))
            n_frames += len(y); n_runs += 1
            ah = np.zeros(len(y), bool); ak = np.zeros(len(y), bool)
            for i,j in itertools.combinations(range(8), 2):
                viol = yh[:,i] & yh[:,j]; both = y[:,i] & y[:,j]; h = viol & ~both
                k = (AU_NAMES[i], AU_NAMES[j])
                agg_e[k]['viol'] += int(viol.sum()); agg_e[k]['help'] += int(h.sum())
                agg_e[k]['hurt'] += int((viol & both).sum())
                ah |= h; ak |= (viol & both)
                for sb in np.unique(sub[h]): per_sub[str(sb)] += int((sub[h]==sb).sum())
            for i,j in itertools.permutations(range(8), 2):
                viol = yh[:,i] & ~yh[:,j]; tv = y[:,i] & ~y[:,j]
                k = (AU_NAMES[i], AU_NAMES[j])
                agg_i[k]['viol'] += int(viol.sum()); agg_i[k]['help'] += int((viol & ~tv).sum())
                agg_i[k]['hurt'] += int((viol & tv).sum())
            fr_help += int(ah.sum()); fr_hurt += int(ak.sum())

if n_runs == 0:
    print('>>> CHƯA CÓ val npz. Không có kết luận nào ở đây.')
else:
    def rep(agg, title, arrow, n=12):
        print(f'\n=== {title} ===')
        rk = sorted(agg.items(), key=lambda kv: kv[1]['help']-kv[1]['hurt'], reverse=True)
        print(f'{"cặp":<17}{"vi phạm":>10}{"help":>9}{"hurt":>9}{"help-hurt":>11}{"net %":>9}')
        print('-'*65)
        for (a,b),c in rk[:n]:
            net = c['help']-c['hurt']
            print(f'{a+arrow+b:<17}{c["viol"]:>10}{c["help"]:>9}{c["hurt"]:>9}{net:>11}{100*net/n_frames:>8.3f}%')
        print(f'... {len(agg)} cặp, {sum(1 for _,c in agg.items() if c["help"]>c["hurt"])} cặp có help > hurt')
    rep(agg_e, 'LOẠI TRỪ (detector bật cả hai)  <-- TÍN HIỆU CHÍNH', ' - ')
    rep(agg_i, 'KÉO THEO (i bật, j tắt)  <-- chỉ có nghĩa SAU khi lọc lift', ' -> ')

    rn = (fr_help-fr_hurt)/n_frames
    print(f'\n{"="*65}')
    print(f'Đã duyệt {n_runs} run · {n_frames} frame')
    print(f'Frame có >=1 luật SỬA được : {fr_help:>8}  ({100*fr_help/n_frames:.3f}%)')
    print(f'Frame có >=1 luật PHÁ hỏng : {fr_hurt:>8}  ({100*fr_hurt/n_frames:.3f}%)')
    print(f'Dư địa ròng (phân biệt)    : {fr_help-fr_hurt:>8}  ({100*rn:.3f}%)')
    if   rn < 0.01: print('\n>>> NO-GO cho kỳ vọng lớn: dưới 1%. Chạy decoder vẫn được (vài phút)\n    nhưng đừng kỳ vọng quá +0.3 điểm. Chuyển trọng tâm sang chế độ ít nhãn.')
    elif rn < 0.03: print('\n>>> BIÊN GIỚI: 1-3%. Chạy decoder, kỳ vọng +0.2 đến +0.5 điểm.')
    else:           print('\n>>> CÓ CHỖ CHO LUẬT. Dựng bảng luật và chạy decoder.')

    if per_sub:
        print('\n--- help theo subject ---')
        tot = sum(per_sub.values())
        for sb,c in sorted(per_sub.items(), key=lambda kv:-kv[1])[:9]:
            print(f'  {sb:<10} {c:>8}  ({100*c/tot:.1f}%)')
        if max(per_sub.values())/tot > 0.35:
            print(f'\n  CẢNH BÁO: một subject chiếm {100*max(per_sub.values())/tot:.0f}% — kết luận không ổn định.')

## Gửi lại gì

Toàn bộ output bốn mục. Quan trọng nhất theo thứ tự:

1. **Mục A** — số hạng nào phân kỳ. Quyết định sửa hàm loss hay sửa quá khớp.
2. **Mục B** — augmentation và scheduler. Quyết định can thiệp nào chặn được quá khớp.
3. **Mục D** — dư địa ròng. Quyết định đầu tư bao nhiêu vào Tầng 2.
4. **Mục C** — xác nhận 9 file val dùng được.